#### ***Rag Citations***
#### ***Rag Citations means includes the external information like page, and source.***

In [1]:
# Load environment variable
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
###create a embedding by using langchain hugging face.

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [3]:
### Create LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001DD1EE7CB90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001DD1EE7C4A0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [4]:
## Load the vector store from Chroma
from langchain_chroma import Chroma
vectorstore = Chroma(
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name = "kubernetes_rag",
    embedding_function=embedding_model
)

In [5]:
vectorstore._collection.count()

9507

In [6]:
### Similarity search
retriever = vectorstore.as_retriever(search_kwargs = {"k":3})

In [7]:
user_query ="What is a Kubernetes Deployment?"

In [13]:
### Test the retriever
retrieved_docs = retriever.invoke(user_query)

print(retrieved_docs)

[Document(id='e4a8d10a-aea4-4e18-8e23-feff97928e35', metadata={'author': '', 'keywords': '', 'creationdate': '', 'subject': '', 'trapped': '', 'total_pages': 676, 'creator': '', 'file_path': '..\\kubernetes\\Concepts.pdf', 'moddate': '', 'source': '..\\kubernetes\\Concepts.pdf', 'modDate': '', 'title': '', 'page': 1, 'creationDate': '', 'producer': 'WeasyPrint 56.1', 'format': 'PDF 1.7'}, page_content='Kubernetes is a portable, extensible, open source platform for managing containerized\nworkloads and services, that facilitates both declarative configuration and automation. It has a\nlarge, rapidly growing ecosystem. Kubernetes services, support, and tools are widely available.\nThe name Kubernetes originates from Greek, meaning helmsman or pilot. K8s as an\nabbreviation results from counting the eight letters between the "K" and the "s". Google open-\nsourced the Kubernetes project in 2014. Kubernetes combines over 15 years of Google\'s\nexperience running production workloads at scal

In [17]:
context_parts = []

for i,doc in enumerate(retrieved_docs,start=1):

    content = doc.page_content

    source = doc.metadata.get("source","Unknown")

    page = doc.metadata.get("page")

    context_parts.append(
        f""" 
    Document {i}
    Content:{content}
    Source: {source}
    Page:{page} """
    )

context = "\n\n".join(context_parts)

print(context)

 
    Document 1
    Content:Kubernetes is a portable, extensible, open source platform for managing containerized
workloads and services, that facilitates both declarative configuration and automation. It has a
large, rapidly growing ecosystem. Kubernetes services, support, and tools are widely available.
The name Kubernetes originates from Greek, meaning helmsman or pilot. K8s as an
abbreviation results from counting the eight letters between the "K" and the "s". Google open-
sourced the Kubernetes project in 2014. Kubernetes combines over 15 years of Google's
experience running production workloads at scale with best-of-breed ideas and practices from
the community.
Going back in time
Let's take a look at why Kubernetes is so useful by going back in time.
Deployment evolution
Traditional deployment era: Early on, organizations ran applications on physical servers.
There was no way to define resource boundaries for applications in a physical server, and this
    Source: ..\kubernetes\

In [21]:
##Design a Prompt
from langchain_core.prompts import ChatPromptTemplate
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


prompt = ChatPromptTemplate.from_template("""
You are a Kubernetes documentation Assistant.

Answer the question using only the provided Context only.


Rules:
    1.Don't use information outside the Context.
    2.If the answer is not available in the context,
    say I don't have information based on the Provided Documents
    3.After the answer provide the source used.
    4.Include the pdf file name and page number don't provide extra information

Context:
{context}

Question:
{question}

Answer:
""")

In [22]:
message = prompt.invoke({"question":user_query,
"context":context

})

response = llm.invoke(message)
print(response.content)

A Kubernetes Deployment is a Kubernetes object that represents an application running on a cluster. It defines the desired state of that application—such as the number of replica pods—through a Deployment spec. The Kubernetes control plane reads this spec, creates the specified number of pod instances, and continuously monitors the actual status. If any pod fails or the actual state diverges from the spec, the system automatically corrects it by, for example, starting replacement pods. Deployments are used to create and update instances of an application in a declarative manner.

**Sources**
- Tutorials.pdf, page 9  
- Concepts.pdf, page 5
